# Discrete Factor

A factor is a function over a collection of discrete variables: $$\phi(X_1,X_2,X_3,...,X_n) \rightarrow \mathbb{R}$$

It assigns one numerical value to every possible assignment of its variables.



In [17]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import dataclass
from itertools import product
from typing import Hashable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np

In [18]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import product
from typing import Hashable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np


@dataclass
class Factor:
    """
    A numerical factor over one or more discrete variables.

    The axis order of `values` follows the order in `variables`.
    """

    variables: Sequence[str]
    domains: Mapping[str, Sequence[Hashable]]
    values: np.ndarray
    name: str = "φ"

    def __post_init__(self) -> None:
        self.variables = list(self.variables)

        self.domains = {
            variable: np.asarray(domain, dtype=object)
            for variable, domain in self.domains.items()
        }

        self.values = np.asarray(
            self.values,
            dtype=float,
        )

        self._validate_parameters()
        self._build_index_maps()

    def _validate_parameters(self) -> None:
        if len(self.variables) == 0:
            raise ValueError(
                "A factor must contain at least one variable."
            )

        if len(set(self.variables)) != len(self.variables):
            raise ValueError(
                "Variable names must be unique."
            )

        if any(
            not isinstance(variable, str)
            for variable in self.variables
        ):
            raise TypeError(
                "Every variable name must be a string."
            )

        missing_domains = [
            variable
            for variable in self.variables
            if variable not in self.domains
        ]

        if missing_domains:
            raise ValueError(
                "Domains are missing for the following variables: "
                f"{missing_domains}."
            )

        for variable in self.variables:
            domain = self.domains[variable]

            if domain.ndim != 1:
                raise ValueError(
                    f"The domain of {variable!r} must be "
                    "one-dimensional."
                )

            if len(domain) == 0:
                raise ValueError(
                    f"The domain of {variable!r} cannot be empty."
                )

            try:
                unique_values = set(domain)
            except TypeError as error:
                raise TypeError(
                    f"Every value in the domain of {variable!r} "
                    "must be hashable."
                ) from error

            if len(unique_values) != len(domain):
                raise ValueError(
                    f"The domain of {variable!r} must contain "
                    "unique values."
                )

        expected_shape = tuple(
            len(self.domains[variable])
            for variable in self.variables
        )

        if self.values.shape != expected_shape:
            raise ValueError(
                "The shape of the factor values must match "
                "the variable domains. "
                f"Expected {expected_shape}, but received "
                f"{self.values.shape}."
            )

        if not np.all(np.isfinite(self.values)):
            raise ValueError(
                "Factor values must contain only finite numbers."
            )

    def _build_index_maps(self) -> None:
        self._value_to_index = {
            variable: {
                value: index
                for index, value in enumerate(
                    self.domains[variable]
                )
            }
            for variable in self.variables
        }

        self._variable_to_axis = {
            variable: axis
            for axis, variable in enumerate(self.variables)
        }

    @property
    def scope(self) -> tuple[str, ...]:
        """Return the ordered variables contained in the factor."""

        return tuple(self.variables)

    @property
    def size(self) -> int:
        """Return the total number of assignments."""

        return int(self.values.size)

    def assignment_to_index(
        self,
        assignment: Mapping[str, Hashable],
    ) -> tuple[int, ...]:
        """Convert an assignment to a NumPy index tuple."""

        missing_variables = [
            variable
            for variable in self.variables
            if variable not in assignment
        ]

        if missing_variables:
            raise ValueError(
                "The assignment is missing values for: "
                f"{missing_variables}."
            )

        unknown_variables = [
            variable
            for variable in assignment
            if variable not in self.variables
        ]

        if unknown_variables:
            raise ValueError(
                "The assignment contains variables that are "
                f"not in the factor: {unknown_variables}."
            )

        indices = []

        for variable in self.variables:
            value = assignment[variable]

            if value not in self._value_to_index[variable]:
                raise ValueError(
                    f"Unknown value {value!r} for variable "
                    f"{variable!r}."
                )

            indices.append(
                self._value_to_index[variable][value]
            )

        return tuple(indices)

    def get_value(
        self,
        assignment: Mapping[str, Hashable],
    ) -> float:
        """Return the factor value for an assignment."""

        index = self.assignment_to_index(assignment)

        return float(self.values[index])

    def set_value(
        self,
        assignment: Mapping[str, Hashable],
        value: float,
    ) -> None:
        """Set the factor value for an assignment."""

        if not np.isfinite(value):
            raise ValueError(
                "The factor value must be finite."
            )

        index = self.assignment_to_index(assignment)

        self.values[index] = float(value)

    def assignments(self):
        """Yield every possible variable assignment."""

        ordered_domains = [
            self.domains[variable]
            for variable in self.variables
        ]

        for values in product(*ordered_domains):
            yield dict(
                zip(
                    self.variables,
                    values,
                )
            )

    def items(self):
        """Yield assignments and their factor values."""

        for assignment in self.assignments():
            yield assignment, self.get_value(assignment)

    def copy(self) -> Factor:
        """Return an independent copy of the factor."""

        return Factor(
            variables=self.variables.copy(),
            domains={
                variable: domain.copy()
                for variable, domain in self.domains.items()
            },
            values=self.values.copy(),
            name=self.name,
        )

    def normalize(self) -> Factor:
        """Return a normalized copy of the factor."""

        if np.any(self.values < 0.0):
            raise ValueError(
                "A factor containing negative values cannot be "
                "normalized as a probability distribution."
            )

        total = self.values.sum()

        if np.isclose(total, 0.0):
            raise ValueError(
                "A factor whose values sum to zero cannot be "
                "normalized."
            )

        return Factor(
            variables=self.variables.copy(),
            domains={
                variable: domain.copy()
                for variable, domain in self.domains.items()
            },
            values=self.values / total,
            name=f"normalized({self.name})",
        )

    def is_normalized(
        self,
        tolerance: float = 1e-9,
    ) -> bool:
        """Check whether the factor is a probability table."""

        if np.any(self.values < 0.0):
            return False

        return bool(
            np.isclose(
                self.values.sum(),
                1.0,
                atol=tolerance,
            )
        )

    def print_table(self) -> None:
        """Print the factor in tabular form."""

        header = [
            *self.variables,
            self.name,
        ]

        rows = []

        for assignment, value in self.items():
            row = [
                assignment[variable]
                for variable in self.variables
            ]

            row.append(value)
            rows.append(row)

        column_widths = []

        for column_index, column_name in enumerate(header):
            maximum_content_width = max(
                len(str(row[column_index]))
                for row in rows
            )

            column_widths.append(
                max(
                    len(str(column_name)),
                    maximum_content_width,
                )
            )

        print(
            " | ".join(
                str(value).ljust(width)
                for value, width in zip(
                    header,
                    column_widths,
                )
            )
        )

        print(
            "-+-".join(
                "-" * width
                for width in column_widths
            )
        )

        for row in rows:
            print(
                " | ".join(
                    str(value).ljust(width)
                    for value, width in zip(
                        row,
                        column_widths,
                    )
                )
            )
    def _validate_shared_domains(
        self,
        other: Factor,
    ) -> None:
        """Check that shared variables have identical domains."""
    
        shared_variables = set(self.variables) & set(
            other.variables
        )
    
        for variable in shared_variables:
            self_domain = self.domains[variable]
            other_domain = other.domains[variable]
    
            if not np.array_equal(
                self_domain,
                other_domain,
            ):
                raise ValueError(
                    f"Shared variable {variable!r} has "
                    "incompatible domains."
                )
    def multiply(
        self,
        other: Factor,
    ) -> Factor:
        """
        Multiply this factor with another factor.
    
        The resulting scope is the ordered union of both scopes.
        """
    
        if not isinstance(other, Factor):
            raise TypeError(
                "A factor can only be multiplied by another Factor."
            )
    
        self._validate_shared_domains(other)
    
        result_variables = list(self.variables)
    
        for variable in other.variables:
            if variable not in result_variables:
                result_variables.append(variable)
    
        result_domains = {}
    
        for variable in result_variables:
            if variable in self.domains:
                result_domains[variable] = (
                    self.domains[variable].copy()
                )
            else:
                result_domains[variable] = (
                    other.domains[variable].copy()
                )
    
        result_shape = tuple(
            len(result_domains[variable])
            for variable in result_variables
        )
    
        result_values = np.empty(
            result_shape,
            dtype=float,
        )
    
        ordered_domains = [
            result_domains[variable]
            for variable in result_variables
        ]
    
        for assignment_values in product(*ordered_domains):
            complete_assignment = dict(
                zip(
                    result_variables,
                    assignment_values,
                )
            )
    
            self_assignment = {
                variable: complete_assignment[variable]
                for variable in self.variables
            }
    
            other_assignment = {
                variable: complete_assignment[variable]
                for variable in other.variables
            }
    
            self_value = self.get_value(self_assignment)
            other_value = other.get_value(other_assignment)
    
            result_assignment_index = tuple(
                np.where(
                    result_domains[variable]
                    == complete_assignment[variable]
                )[0][0]
                for variable in result_variables
            )
    
            result_values[result_assignment_index] = (
                self_value * other_value
            )
    
        return Factor(
            variables=result_variables,
            domains=result_domains,
            values=result_values,
            name=f"({self.name} × {other.name})",
        )
    def __mul__(
        self,
        other: Factor,
    ) -> Factor:
        """Return the product of two factors."""
    
        return self.multiply(other)

    def marginalize(
        self,
        variable: str,
    ) -> Factor | float:
        """
        Sum out one variable from the factor.
    
        Parameters
        ----------
        variable
            Variable to eliminate.
    
        Returns
        -------
        Factor | float
            Reduced factor, or a scalar if no variables remain.
        """
        if variable not in self.variables:
            raise ValueError(
                f"Variable {variable!r} is not in the factor."
            )
    
        axis = self._variable_to_axis[variable]
        result_values = self.values.sum(axis=axis)
    
        result_variables = [
            current_variable
            for current_variable in self.variables
            if current_variable != variable
        ]
    
        if not result_variables:
            return float(result_values)
    
        result_domains = {
            current_variable: self.domains[
                current_variable
            ].copy()
            for current_variable in result_variables
        }
    
        return Factor(
            variables=result_variables,
            domains=result_domains,
            values=result_values,
            name=f"Σ_{variable}({self.name})",
        )
        
    def reduce(
        self,
        evidence: Mapping[str, Hashable],
    ) -> Factor:
        """
        Fix one or more variables to observed values.
    
        Parameters
        ----------
        evidence:''
            Mapping from observed variables to their values.
        """
    
        unknown_variables = [
            variable
            for variable in evidence
            if variable not in self.variables
        ]
    
        if unknown_variables:
            raise ValueError(
                "Evidence contains variables not present in "
                f"the factor: {unknown_variables}."
            )
    
        indexing = []
    
        for variable in self.variables:
            if variable in evidence:
                observed_value = evidence[variable]
    
                if (
                    observed_value
                    not in self._value_to_index[variable]
                ):
                    raise ValueError(
                        f"Unknown value {observed_value!r} for "
                        f"variable {variable!r}."
                    )
    
                indexing.append(
                    self._value_to_index[variable][
                        observed_value
                    ]
                )
            else:
                indexing.append(slice(None))
    
        result_values = self.values[tuple(indexing)]
    
        result_variables = [
            variable
            for variable in self.variables
            if variable not in evidence
        ]
    
        result_domains = {
            variable: self.domains[variable].copy()
            for variable in result_variables
        }
    
        if len(result_variables) == 0:
            return float(result_values)
    
        return Factor(
            variables=result_variables,
            domains=result_domains,
            values=result_values,
            name=f"{self.name} | {dict(evidence)}",
        )

    def marginalize_many(
        self,
        variables: Sequence[str],
    ) -> Factor:
        """Sum out several variables sequentially."""
    
        result = self
    
        for variable in variables:
            result = result.marginalize(variable)
    
        return result
        
    def plot_heatmap(self) -> None:
        """Plot a two-variable factor as a heatmap."""

        if len(self.variables) != 2:
            raise ValueError(
                "A heatmap requires a factor with exactly "
                "two variables."
            )

        row_variable = self.variables[0]
        column_variable = self.variables[1]

        fig, ax = plt.subplots(figsize=(7, 5))

        image = ax.imshow(self.values)

        ax.set_xticks(
            np.arange(
                len(self.domains[column_variable])
            )
        )

        ax.set_yticks(
            np.arange(
                len(self.domains[row_variable])
            )
        )

        ax.set_xticklabels(
            self.domains[column_variable]
        )

        ax.set_yticklabels(
            self.domains[row_variable]
        )

        ax.set_xlabel(column_variable)
        ax.set_ylabel(row_variable)
        ax.set_title(f"Factor {self.name}")

        for index in np.ndindex(self.values.shape):
            row_index, column_index = index

            ax.text(
                column_index,
                row_index,
                f"{self.values[index]:.3f}",
                ha="center",
                va="center",
            )

        fig.colorbar(
            image,
            ax=ax,
            label="Factor value",
        )

        plt.tight_layout()
        plt.show()

In [19]:
factor = Factor(
    variables=[
        "Weather",
        "Traffic",
    ],
    domains={
        "Weather": [
            "sunny",
            "rainy",
        ],
        "Traffic": [
            "light",
            "heavy",
        ],
    },
    values=[
        [5.0, 2.0],
        [3.0, 8.0],
    ],
    name="φ(W,T)",
)

In [20]:
print("Scope:", factor.scope)
print("Shape:", factor.values.shape)
print("Number of entries:", factor.size)
print("Normalized:", factor.is_normalized())

Scope: ('Weather', 'Traffic')
Shape: (2, 2)
Number of entries: 4
Normalized: False


In [21]:
factor.print_table()

Weather | Traffic | φ(W,T)
--------+---------+-------
sunny   | light   | 5.0   
sunny   | heavy   | 2.0   
rainy   | light   | 3.0   
rainy   | heavy   | 8.0   


In [22]:
normalized_factor = factor.normalize()

normalized_factor.print_table()

Weather | Traffic | normalized(φ(W,T)) 
--------+---------+--------------------
sunny   | light   | 0.2777777777777778 
sunny   | heavy   | 0.1111111111111111 
rainy   | light   | 0.16666666666666666
rainy   | heavy   | 0.4444444444444444 


In [23]:
factor_ab = Factor(
    variables=["A", "B"],
    domains={
        "A": [0, 1],
        "B": [0, 1],
    },
    values=[
        [0.3, 0.7],
        [0.8, 0.2],
    ],
    name="φ₁(A,B)",
)

In [24]:
factor_bc = Factor(
    variables=["B", "C"],
    domains={
        "B": [0, 1],
        "C": [0, 1],
    },
    values=[
        [0.9, 0.1],
        [0.4, 0.6],
    ],
    name="φ₂(B,C)",
)

In [25]:
factor_abc = factor_ab.multiply(factor_bc)

print("Scope:", factor_abc.scope)
print("Shape:", factor_abc.values.shape)

factor_abc.print_table()

Scope: ('A', 'B', 'C')
Shape: (2, 2, 2)
A | B | C | (φ₁(A,B) × φ₂(B,C))
--+---+---+--------------------
0 | 0 | 0 | 0.27               
0 | 0 | 1 | 0.03               
0 | 1 | 0 | 0.27999999999999997
0 | 1 | 1 | 0.42               
1 | 0 | 0 | 0.7200000000000001 
1 | 0 | 1 | 0.08000000000000002
1 | 1 | 0 | 0.08000000000000002
1 | 1 | 1 | 0.12               


In [26]:
factor_abc = factor_ab * factor_bc

In [27]:
factor_ac = factor_abc.marginalize("B")

print("Scope:", factor_ac.scope)
print("Shape:", factor_ac.values.shape)

factor_ac.print_table()

Scope: ('A', 'C')
Shape: (2, 2)
A | C | Σ_B((φ₁(A,B) × φ₂(B,C)))
--+---+-------------------------
0 | 0 | 0.55                    
0 | 1 | 0.44999999999999996     
1 | 0 | 0.8                     
1 | 1 | 0.2                     


In [28]:
factor_ac_given_b1 = factor_abc.reduce(
    {"B": 1}
)

factor_ac_given_b1.print_table()

A | C | (φ₁(A,B) × φ₂(B,C)) | {'B': 1}
--+---+-------------------------------
0 | 0 | 0.27999999999999997           
0 | 1 | 0.42                          
1 | 0 | 0.08000000000000002           
1 | 1 | 0.12                          


In [29]:
factor_a = factor_abc.marginalize_many(
    ["B", "C"]
)

factor_a.print_table()

A | Σ_C(Σ_B((φ₁(A,B) × φ₂(B,C))))
--+------------------------------
0 | 1.0                          
1 | 1.0                          


In [30]:
factor_a = Factor(
    variables=["A"],
    domains={
        "A": [0, 1],
    },
    values=[
        0.3,
        0.7,
    ],
    name="φ₁(A)",
)

factor_c = Factor(
    variables=["C"],
    domains={
        "C": ["low", "high"],
    },
    values=[
        2.0,
        5.0,
    ],
    name="φ₂(C)",
)

In [31]:
factor_ac = factor_a * factor_c

print("Scope:", factor_ac.scope)
print("Shape:", factor_ac.values.shape)

factor_ac.print_table()

Scope: ('A', 'C')
Shape: (2, 2)
A | C    | (φ₁(A) × φ₂(C))
--+------+----------------
0 | low  | 0.6            
0 | high | 1.5            
1 | low  | 1.4            
1 | high | 3.5            


In [32]:
expected_values = np.array([
    [0.6, 1.5],
    [1.4, 3.5],
])

print(factor_ac.values)

[[0.6 1.5]
 [1.4 3.5]]
